# 13 Rag Evaluation Ragas

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `13-rag-evaluation-ragas.ipynb`

In [2]:
# # Start coding here
# !pip install ragas

In [4]:
# ==================================================
# Notebook 13
# RAG Evaluation
# ==================================================

import pandas as pd
import numpy as np

from datasets import Dataset

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [5]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [6]:
evaluation_data = [
    {
        "question": "What was revenue in 2026?",
        "answer": "Revenue was $5.2 billion.",
        "contexts": ["Revenue reached $5.2 billion in 2026."],
        "ground_truth": "Revenue was $5.2 billion.",
    },
    {
        "question": "Did revenue increase?",
        "answer": "Revenue increased by 12%.",
        "contexts": ["Revenue increased by 12%."],
        "ground_truth": "Revenue increased by 12%.",
    },
]

In [7]:
df = pd.DataFrame(evaluation_data)

df.head()

,question,answer,contexts,ground_truth
0,What was revenue in 2026?,Revenue was $5.2 billion.,[Revenue reached $5.2 billion in 2026.],Revenue was $5.2 billion.
1,Did revenue increase?,Revenue increased by 12%.,[Revenue increased by 12%.],Revenue increased by 12%.


In [8]:
dataset = Dataset.from_pandas(df)

dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 2
})

In [9]:
def answer_relevancy(question, answer):

    q_emb = embedding_model.encode(question)

    a_emb = embedding_model.encode(answer)

    score = cosine_similarity(q_emb.reshape(1, -1), a_emb.reshape(1, -1))[0][0]

    return float(score)

In [10]:
answer_relevancy("What was revenue?", "Revenue was $5.2 billion.")

0.7129266262054443

In [11]:
def faithfulness(answer, contexts):

    context_text = " ".join(contexts)

    answer_emb = embedding_model.encode(answer)

    context_emb = embedding_model.encode(context_text)

    score = cosine_similarity(answer_emb.reshape(1, -1), context_emb.reshape(1, -1))[0][
        0
    ]

    return float(score)

In [12]:
faithfulness("Revenue was $5.2 billion.", ["Revenue reached $5.2 billion in 2026."])

0.859982967376709

In [13]:
def context_precision(question, contexts):

    question_emb = embedding_model.encode(question)

    scores = []

    for context in contexts:

        context_emb = embedding_model.encode(context)

        similarity = cosine_similarity(
            question_emb.reshape(1, -1), context_emb.reshape(1, -1)
        )[0][0]

        scores.append(similarity)

    return float(np.mean(scores))

In [14]:
context_precision("What was revenue?", ["Revenue reached $5.2 billion."])

0.6713289022445679

In [15]:
def context_recall(ground_truth, contexts):

    context_text = " ".join(contexts)

    gt_emb = embedding_model.encode(ground_truth)

    context_emb = embedding_model.encode(context_text)

    score = cosine_similarity(gt_emb.reshape(1, -1), context_emb.reshape(1, -1))[0][0]

    return float(score)

In [16]:
context_recall("Revenue was $5.2 billion.", ["Revenue reached $5.2 billion."])

0.9507078528404236

In [17]:
results = []

In [18]:
for row in evaluation_data:

    metrics = {
        "question": row["question"],
        "answer_relevancy": answer_relevancy(row["question"], row["answer"]),
        "faithfulness": faithfulness(row["answer"], row["contexts"]),
        "context_precision": context_precision(row["question"], row["contexts"]),
        "context_recall": context_recall(row["ground_truth"], row["contexts"]),
    }

    results.append(metrics)

In [19]:
results_df = pd.DataFrame(results)

results_df

,question,answer_relevancy,faithfulness,context_precision,context_recall
0,What was revenue in 2026?,0.687975,0.859983,0.801002,0.859983
1,Did revenue increase?,0.799108,1.000000,0.799108,1.000000


In [20]:
results_df.mean(numeric_only=True)

answer_relevancy     0.743542
faithfulness         0.929991
context_precision    0.800055
context_recall       0.929991
dtype: float64

In [21]:
bad_case = {
    "question": "What was revenue?",
    "answer": "Revenue was $7 billion.",
    "contexts": ["Revenue reached $5.2 billion."],
    "ground_truth": "Revenue was $5.2 billion.",
}

In [22]:
faithfulness(bad_case["answer"], bad_case["contexts"])

0.7460458874702454

In [23]:
def detect_hallucination(answer, contexts, threshold=0.75):

    score = faithfulness(answer, contexts)

    return score < threshold

In [24]:
detect_hallucination(bad_case["answer"], bad_case["contexts"])

True

In [25]:
def retrieval_failure(question, contexts, threshold=0.60):

    score = context_precision(question, contexts)

    return score < threshold

In [31]:
question = bad_case["question"]

contexts = bad_case["contexts"]

retrieval_failure(question, contexts)

False

In [32]:
def retrieval_failure(
    question,
    contexts,
    threshold=0.60,
):
    score = context_precision(
        question,
        contexts,
    )

    print(f"Context Precision: {score:.4f}")

    return score < threshold

In [33]:
retrieval_failure(
    "What was revenue in 2026?", ["Revenue reached $5.2 billion in 2026."]
)

Context Precision: 0.8010


False

In [ ]:
# from ragas import evaluate

# print("RAGAS OK")

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [34]:
import ragas
import langchain
import langchain_community

print("ragas:", ragas.__version__)
print("langchain:", langchain.__version__)
print("langchain-community:", langchain_community.__version__)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [29]:
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'